In [1]:
import os, sys, importlib, csv, time, traceback
from pathlib import Path
from datetime import datetime
from collections import Counter

import pandas as pd

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_TRACING"]    = "false"

# ── مسیرها ────────────────────────────────────────────────
MA_ROOT      = Path(r"F:\Thesis\project\3-Multi-Agent-System\Test_402\MA")
PROJECT_ROOT = MA_ROOT / "legal_multi_agent"
NOTEBOOK_DIR = Path(r"F:\Thesis\project\3-Multi-Agent-System\Test_402\MA")

for p in [str(MA_ROOT), str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print(f"✓ MA_ROOT         : {MA_ROOT}")
print(f"✓ PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"✓ agents exists   : {(PROJECT_ROOT / 'agents').exists()}")
print(f"✓ graphs exists   : {(PROJECT_ROOT / 'graphs').exists()}")
print(f"✓ OPENROUTER_API_KEY: {(os.getenv('OPENROUTER_API_KEY') or '')[:15]} ...")
print(f"✓ COHERE_API_KEY    : {(os.getenv('COHERE_API_KEY')     or '')[:15]} ...")
print(f"✓ MODEL             : {os.getenv('MODEL', '⚠️ تعریف نشده!')}")

✓ MA_ROOT         : F:\Thesis\project\3-Multi-Agent-System\Test_402\MA
✓ PROJECT_ROOT    : F:\Thesis\project\3-Multi-Agent-System\Test_402\MA\legal_multi_agent
✓ agents exists   : True
✓ graphs exists   : True
✓ OPENROUTER_API_KEY: sk-or-v1-4bcfd4 ...
✓ COHERE_API_KEY    : O0pIM8Zkm973sO7 ...
✓ MODEL             : qwen/qwen3-next-80b-a3b-instruct


In [2]:
modules_to_reload = [
    'legal_multi_agent.utils.toon',
    'legal_multi_agent.state.schemas',
    'legal_multi_agent.agents.supervisor',
    'legal_multi_agent.agents.researcher',
    'legal_multi_agent.agents.reasoner',
    'legal_multi_agent.agents.critic',
    'legal_multi_agent.agents.tool_executor',
    'legal_multi_agent.graphs.main_graph',
]

print("🔄 Reloading modules...")
for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"  ♻️  Reloaded : {module_name}")
    else:
        print(f"  ⬇️  Loading  : {module_name}")

print("✅ All modules reloaded\n")

from legal_multi_agent.graphs.main_graph import graph
print("✅ Graph imported")

🔄 Reloading modules...
  ⬇️  Loading  : legal_multi_agent.utils.toon
  ⬇️  Loading  : legal_multi_agent.state.schemas
  ⬇️  Loading  : legal_multi_agent.agents.supervisor
  ⬇️  Loading  : legal_multi_agent.agents.researcher
  ⬇️  Loading  : legal_multi_agent.agents.reasoner
  ⬇️  Loading  : legal_multi_agent.agents.critic
  ⬇️  Loading  : legal_multi_agent.agents.tool_executor
  ⬇️  Loading  : legal_multi_agent.graphs.main_graph
✅ All modules reloaded

✅ Graph imported


In [3]:
# ── مسیر فایل‌های CSV ─────────────────────────────────────
QUESTIONS_CSV = r"F:\Thesis\project\403-vekalat\structured_questions_402.csv"
GOLD_CSV      = r"F:\Thesis\project\3-Multi-Agent-System\Test_402\BaseLine\Gold_402\Gold_402.csv"

df_questions = pd.read_csv(QUESTIONS_CSV, encoding="utf-8-sig")
df_gold      = pd.read_csv(GOLD_CSV,      encoding="utf-8-sig")

# ── یکپارچه‌سازی ──────────────────────────────────────────
df_gold = df_gold.rename(columns={"idx": "question_number"})
df = df_questions.merge(df_gold[["question_number", "Gold", "explain"]],
                        on="question_number", how="left")

print(f"✅ سؤالات بارگذاری شد: {len(df)} سؤال")
print(f"   ستون‌ها: {list(df.columns)}")
print("\n📋 نمونه داده:")
print(df[["question_number", "Gold", "question"]].head(3).to_string(index=False))

✅ سؤالات بارگذاری شد: 10 سؤال
   ستون‌ها: ['question_number', 'category', 'question', 'options', 'Gold', 'explain']

📋 نمونه داده:
 question_number  Gold                                                                                                                                                                                                                                                              question
               1     3 «الف» به وکالت از «ب»، ملک «ب» را به مبلغ معین به «ج» می‌فروشد و ثمن معامله نیز از بابت بدهی سابق «الف» به «ج» در قالب عقد حواله‌ای که به تأیید «ب» می‌رسد، کارسازی می‌شود. در صورت فسخ بعدی عقد بیع به علت خیار غبن از ناحیه «ج»، تکلیف ثمن دریافتی چگونه خواهد بود؟
               2     1                                                                                                                                                                                                                               درباره «دلیل لبّی»، کدام مورد صحیح است؟
              

In [4]:
# ── ستون‌های مکالمه (برای هر sheet) ──────────────────────
CONV_COLUMNS = [
    "step", "node", "agent", "role", "content_type",
    "content", "meta_answer", "meta_needs_revision",
    "meta_issue", "meta_action", "meta_reasoning",
]

def make_add_row(rows_list):
    """یک تابع add_row برای یک لیست rows مستقل می‌سازد"""
    def add_row(step, node, agent, role, content_type, content,
                meta_answer=None, meta_needs_revision=None,
                meta_issue=None, meta_action=None, meta_reasoning=None):
        rows_list.append({
            "step":               step,
            "node":               node,
            "agent":              agent,
            "role":               role,
            "content_type":       content_type,
            "content":            str(content)            if content            is not None else "",
            "meta_answer":        str(meta_answer)        if meta_answer        is not None else "",
            "meta_needs_revision":str(meta_needs_revision)if meta_needs_revision is not None else "",
            "meta_issue":         str(meta_issue)         if meta_issue         is not None else "",
            "meta_action":        str(meta_action)        if meta_action        is not None else "",
            "meta_reasoning":     str(meta_reasoning)     if meta_reasoning     is not None else "",
        })
    return add_row


def run_one_question(q_num, question, options_text,
                     use_option_verifier=True,
                     use_retriever_tool=False,
                     max_revisions=2):
    """
    یک سؤال را اجرا می‌کند و dict نتیجه + لیست rows مکالمه را برمی‌گرداند.
    """
    conv_rows = []
    add_row   = make_add_row(conv_rows)

    initial_state = {
        "question_number":     q_num,
        "question":            question,
        "options_text":        options_text,
        "max_revisions":       max_revisions,
        "use_option_verifier": use_option_verifier,
        "use_retriever_tool":  use_retriever_tool,
    }

    full_state          = dict(initial_state)
    prev_messages_count = 0
    prev_draft_toon     = None
    prev_critic_toon    = None
    prev_final_toon     = None
    step_times          = {}
    t_global_start      = time.time()

    print(f"\n{'═'*65}")
    print(f"🚀 سؤال {q_num:>2}  |  {question[:55]}...")
    print(f"{'═'*65}")

    try:
        for step_num, state_snapshot in enumerate(
            graph.stream(initial_state, {"recursion_limit": 60}), start=1
        ):
            node_name = list(state_snapshot.keys())[0]
            delta     = state_snapshot[node_name]
            t_elapsed = round(time.time() - t_global_start, 2)

            for k, v in delta.items():
                if k == "messages" and isinstance(v, list):
                    existing = full_state.get("messages") or []
                    full_state["messages"] = existing + [m for m in v if m not in existing]
                else:
                    full_state[k] = v

            messages     = full_state.get("messages") or []
            new_messages = messages[prev_messages_count:]
            prev_messages_count = len(messages)

            current_draft  = full_state.get("draft_toon")
            current_critic = full_state.get("critic_toon")
            current_final  = full_state.get("final_toon")

            new_draft  = current_draft  if current_draft  != prev_draft_toon  else None
            new_critic = current_critic if current_critic != prev_critic_toon else None
            new_final  = current_final  if current_final  != prev_final_toon  else None

            prev_draft_toon  = current_draft
            prev_critic_toon = current_critic
            prev_final_toon  = current_final
            step_times[f"step_{step_num}_{node_name}"] = t_elapsed

            # ── CSV: state summary ─────────────────────────
            add_row(step_num, node_name.upper(), node_name, "system", "state_summary",
                    f"next={full_state.get('next')} | ctx={len(full_state.get('context') or '')}chars"
                    f" | n_docs={len(full_state.get('rag_results') or [])} | msgs={len(messages)}"
                    f" | elapsed={t_elapsed}s")

            # ── چاپ خلاصه step ────────────────────────────
            print(f"  📍 Step {step_num:>2} [{node_name.upper():<12}] "
                  f"next={str(full_state.get('next') or ''):<12} "
                  f"msgs={len(messages):>2} | {t_elapsed}s")

            # ── پیام‌های جدید ──────────────────────────────
            for msg in new_messages:
                if not isinstance(msg, dict):
                    continue
                meta    = msg.get("metadata") or {}
                agent   = meta.get("agent") or msg.get("name") or msg.get("role") or "?"
                role    = msg.get("role", "?")
                content = msg.get("content") or ""
                add_row(step_num, node_name.upper(), agent.upper(), role,
                        "message", content)

            # ── RAG docs ───────────────────────────────────
            if node_name.upper() == "RESEARCHER":
                for i, doc in enumerate(full_state.get("rag_results") or [], 1):
                    if isinstance(doc, dict):
                        meta_d = doc.get("metadata") or {}
                        title  = meta_d.get("title") or meta_d.get("source") or f"سند {i}"
                        add_row(step_num, "RESEARCHER", "RAG", "system",
                                f"doc_{i}_content", doc.get("content") or doc.get("page_content") or "")

            # ── Draft جدید ─────────────────────────────────
            if new_draft:
                explanation = new_draft.get("explanation") or ""
                print(f"       ✏️  DRAFT  → گزینه {new_draft.get('answer')}")
                add_row(step_num, node_name.upper(), "REASONER", "assistant",
                        "draft", explanation,
                        meta_answer=new_draft.get("answer"))

            # ── Critic جدید ────────────────────────────────
            if new_critic:
                icon = "🟢" if not new_critic.get("needs_revision") else "🔴"
                print(f"       {icon} CRITIC → needs_revision={new_critic.get('needs_revision')}")
                reasoning = new_critic.get("reasoning") or ""
                add_row(step_num, node_name.upper(), "CRITIC", "assistant",
                        "critic_verdict", reasoning,
                        meta_needs_revision=new_critic.get("needs_revision"),
                        meta_issue=new_critic.get("issue"),
                        meta_action=new_critic.get("action"),
                        meta_reasoning=reasoning)

            # ── Final جدید ─────────────────────────────────
            if new_final:
                explanation = new_final.get("explanation") or ""
                print(f"       🏁 FINAL  → گزینه {new_final.get('answer')}")
                add_row(step_num, node_name.upper(), "FINALIZE", "system",
                        "final", explanation,
                        meta_answer=new_final.get("answer"))

        # ── نتیجه نهایی ────────────────────────────────────
        final_toon    = full_state.get("final_toon") or {}
        draft_toon    = full_state.get("draft_toon") or {}
        verifier_toon = full_state.get("verifier_toon") or {}

        final_ans    = str(final_toon.get("answer")    or "").strip()
        draft_ans    = str(draft_toon.get("answer")    or "").strip()
        verifier_ans = str(verifier_toon.get("answer") or "").strip()
        revisions    = full_state.get("revision_count", 0)
        total_time   = round(time.time() - t_global_start, 2)
        error_msg    = ""

    except Exception as e:
        final_ans    = ""
        draft_ans    = ""
        verifier_ans = ""
        revisions    = 0
        total_time   = round(time.time() - t_global_start, 2)
        error_msg    = str(e)
        print(f"  ❌ خطا: {error_msg[:120]}")
        traceback.print_exc()

    # ── خلاصه SUMMARY در CSV ───────────────────────────────
    v_match = "✅ موافق" if verifier_ans == final_ans else "⚠️ مخالف"
    d_match = "✅ موافق" if draft_ans    == final_ans else "⚠️ مخالف"
    add_row(0, "SUMMARY", "SYSTEM", "summary", "decision_summary",
            f"verifier={verifier_ans} | draft={draft_ans} | final={final_ans}"
            f" | revisions={revisions} | total_time={total_time}s",
            meta_answer=str(final_ans),
            meta_reasoning=f"v_match={v_match} d_match={d_match} error={error_msg}")

    print(f"  ─── نتیجه: final={final_ans} | verifier={verifier_ans} "
          f"| draft={draft_ans} | rev={revisions} | {total_time}s")

    result = {
        "question_number": q_num,
        "final_answer":    final_ans,
        "draft_answer":    draft_ans,
        "verifier_answer": verifier_ans,
        "revision_count":  revisions,
        "total_time_s":    total_time,
        "error":           error_msg,
    }
    return result, conv_rows


print("✅ توابع کمکی تعریف شدند")

✅ توابع کمکی تعریف شدند


In [5]:
# ── تنظیمات ───────────────────────────────────────────────
USE_OPTION_VERIFIER = True
USE_RETRIEVER_TOOL  = False
MAX_REVISIONS       = 2

# ── پوشه خروجی ────────────────────────────────────────────
OUTPUT_DIR = NOTEBOOK_DIR / "eval_results"
OUTPUT_DIR.mkdir(exist_ok=True)

timestamp       = datetime.now().strftime("%Y%m%d_%H%M%S")
EXCEL_PATH      = OUTPUT_DIR / f"eval_402_{timestamp}.xlsx"

# ── containers ────────────────────────────────────────────
all_results   = []          # یک ردیف به ازای هر سؤال
all_conv_rows = {}          # کلید: q_num  →  لیست ردیف‌های مکالمه

print(f"🎯 شروع ارزیابی {len(df)} سؤال")
print(f"   use_option_verifier: {USE_OPTION_VERIFIER}")
print(f"   use_retriever_tool : {USE_RETRIEVER_TOOL}")
print(f"   max_revisions      : {MAX_REVISIONS}")
print(f"   خروجی Excel        : {EXCEL_PATH}\n")

batch_start = time.time()

for _, row in df.iterrows():
    q_num      = int(row["question_number"])
    question   = str(row["question"])
    options    = str(row["options"])
    gold       = str(row["Gold"]).strip()

    result, conv_rows = run_one_question(
        q_num         = q_num,
        question      = question,
        options_text  = options,
        use_option_verifier = USE_OPTION_VERIFIER,
        use_retriever_tool  = USE_RETRIEVER_TOOL,
        max_revisions       = MAX_REVISIONS,
    )

    result["Gold"]      = gold
    result["correct"]   = (result["final_answer"] == gold)
    result["question"]  = question[:80] + "..."
    result["options"]   = options[:120] + "..."
    result["gold_explain"] = str(row.get("explain") or "")[:300]

    all_results.append(result)
    all_conv_rows[q_num] = conv_rows

    icon = "✅" if result["correct"] else "❌"
    print(f"  {icon}  Q{q_num:>2} | final={result['final_answer']} | gold={gold} | {result['total_time_s']}s\n")

batch_total = round(time.time() - batch_start, 2)
print(f"\n{'═'*65}")
print(f"🏁 ارزیابی تمام شد — زمان کل: {batch_total}s")
print(f"{'═'*65}")

🎯 شروع ارزیابی 10 سؤال
   use_option_verifier: True
   use_retriever_tool : False
   max_revisions      : 2
   خروجی Excel        : F:\Thesis\project\3-Multi-Agent-System\Test_402\MA\eval_results\eval_402_20260627_203148.xlsx


═════════════════════════════════════════════════════════════════
🚀 سؤال  1  |  «الف» به وکالت از «ب»، ملک «ب» را به مبلغ معین به «ج» م...
═════════════════════════════════════════════════════════════════
  📍 Step  1 [INITIALIZE  ] next=             msgs= 0 | 0.0s
  📍 Step  2 [SUPERVISOR  ] next=researcher   msgs= 1 | 0.01s


  📍 Step  3 [RESEARCHER  ] next=researcher   msgs= 2 | 11.23s
  📍 Step  4 [SUPERVISOR  ] next=reasoner     msgs= 3 | 11.24s
  📍 Step  5 [REASONER    ] next=reasoner     msgs= 4 | 11.24s
  📍 Step  6 [SUPERVISOR  ] next=tools        msgs= 5 | 11.24s
  📍 Step  7 [TOOLS       ] next=tools        msgs= 6 | 15.07s
  📍 Step  8 [SUPERVISOR  ] next=reasoner     msgs= 6 | 15.07s
  📍 Step  9 [REASONER    ] next=reasoner     msgs= 7 | 23.09s
       ✏️  DRAFT  → گزینه 3
  📍 Step 10 [SUPERVISOR  ] next=critic       msgs= 8 | 23.09s
  📍 Step 11 [CRITIC      ] next=critic       msgs= 9 | 35.88s
       🟢 CRITIC → needs_revision=False
  📍 Step 12 [SUPERVISOR  ] next=finalize     msgs=10 | 35.88s
  📍 Step 13 [FINALIZE    ] next=finalize     msgs=10 | 35.88s
       🏁 FINAL  → گزینه 3
  📍 Step 14 [SUPERVISOR  ] next=FINISH       msgs=11 | 35.88s
  ─── نتیجه: final=3 | verifier= | draft=3 | rev=0 | 35.88s
  ✅  Q 1 | final=3 | gold=3 | 35.88s


════════════════════════════════════════════════════════════════

In [6]:
df_results = pd.DataFrame(all_results)

correct  = df_results["correct"].sum()
total    = len(df_results)
accuracy = correct / total * 100

print(f"\n{'━'*65}")
print(f"  📊 نتایج ارزیابی — مدل Legal Multi-Agent")
print(f"{'━'*65}")
print(f"  Accuracy کل   : {correct}/{total}  =  {accuracy:.1f}%")
print(f"  زمان میانگین  : {df_results['total_time_s'].mean():.1f}s / سؤال")
print(f"  revision میانگین: {df_results['revision_count'].mean():.2f}")
print(f"{'━'*65}")

print("\n📋 جزئیات هر سؤال:")
for _, r in df_results.iterrows():
    icon = "✅" if r["correct"] else "❌"
    err  = f" [ERR:{r['error'][:40]}]" if r["error"] else ""
    print(f"  {icon} Q{int(r['question_number']):>2} | final={r['final_answer']} | gold={r['Gold']}"
          f" | verifier={r['verifier_answer']} | rev={r['revision_count']} | {r['total_time_s']}s{err}")

print(f"\n✅ نتایج در df_results ذخیره شد")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📊 نتایج ارزیابی — مدل Legal Multi-Agent
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Accuracy کل   : 6/10  =  60.0%
  زمان میانگین  : 40.8s / سؤال
  revision میانگین: 0.00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 جزئیات هر سؤال:
  ✅ Q 1 | final=3 | gold=3 | verifier= | rev=0 | 35.88s
  ❌ Q 2 | final=4 | gold=1 | verifier= | rev=0 | 31.04s
  ✅ Q 3 | final=2 | gold=2 | verifier= | rev=0 | 43.02s
  ✅ Q 4 | final=1 | gold=1 | verifier= | rev=0 | 45.71s
  ✅ Q 5 | final=4 | gold=4 | verifier= | rev=0 | 37.88s
  ❌ Q 6 | final=4 | gold=2 | verifier= | rev=0 | 50.75s
  ❌ Q 7 | final=1 | gold=2 | verifier= | rev=0 | 45.98s
  ✅ Q 8 | final=1 | gold=1 | verifier= | rev=0 | 48.52s
  ✅ Q 9 | final=4 | gold=4 | verifier= | rev=0 | 32.0s
  ❌ Q10 | final=3 | gold=2 | verifier= | rev=0 | 37.7s

✅ نتایج در df_results ذخیره شد


In [7]:
print(f"💾 در حال ذخیره Excel: {EXCEL_PATH}")

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:

    # ══ Sheet 1: خلاصه نتایج ══════════════════════════════
    summary_cols = [
        "question_number", "correct", "final_answer", "Gold",
        "verifier_answer", "draft_answer", "revision_count",
        "total_time_s", "error", "question", "gold_explain",
    ]
    df_summary = df_results[summary_cols].copy()
    df_summary["correct"] = df_summary["correct"].map({True: "✅", False: "❌"})
    df_summary.to_excel(writer, sheet_name="📊 Summary", index=False)

    # ── فرمت‌دهی Summary ────────────────────────────────────
    ws = writer.sheets["📊 Summary"]
    ws.column_dimensions["A"].width = 8
    ws.column_dimensions["B"].width = 8
    ws.column_dimensions["C"].width = 10
    ws.column_dimensions["D"].width = 8
    ws.column_dimensions["E"].width = 10
    ws.column_dimensions["F"].width = 10
    ws.column_dimensions["G"].width = 8
    ws.column_dimensions["H"].width = 10
    ws.column_dimensions["I"].width = 30
    ws.column_dimensions["J"].width = 50
    ws.column_dimensions["K"].width = 60

    # ══ Sheet 2: Accuracy Stats ═══════════════════════════
    correct_count  = df_results["correct"].sum()
    total_count    = len(df_results)
    accuracy_val   = round(correct_count / total_count * 100, 2)
    stats_data = {
        "معیار": [
            "تعداد کل سؤالات",
            "تعداد صحیح",
            "تعداد غلط",
            "Accuracy (%)",
            "میانگین زمان (s)",
            "میانگین Revision",
            "خطاهای اجرایی",
        ],
        "مقدار": [
            total_count,
            int(correct_count),
            int(total_count - correct_count),
            accuracy_val,
            round(df_results["total_time_s"].mean(), 2),
            round(df_results["revision_count"].mean(), 3),
            int(df_results["error"].astype(bool).sum()),
        ]
    }
    df_stats = pd.DataFrame(stats_data)
    df_stats.to_excel(writer, sheet_name="📈 Accuracy", index=False)

    ws2 = writer.sheets["📈 Accuracy"]
    ws2.column_dimensions["A"].width = 30
    ws2.column_dimensions["B"].width = 20

    # ══ Sheet 3+: مکالمه هر سؤال در یک تب جداگانه ════════
    for q_num, rows in all_conv_rows.items():
        sheet_name = f"Q{q_num:02d}_conv"
        if not rows:
            df_empty = pd.DataFrame(columns=CONV_COLUMNS)
            df_empty.to_excel(writer, sheet_name=sheet_name, index=False)
            continue

        df_conv = pd.DataFrame(rows, columns=CONV_COLUMNS)
        df_conv.to_excel(writer, sheet_name=sheet_name, index=False)

        # ── عرض ستون‌ها ─────────────────────────────────
        ws_c = writer.sheets[sheet_name]
        ws_c.column_dimensions["A"].width = 6   # step
        ws_c.column_dimensions["B"].width = 14  # node
        ws_c.column_dimensions["C"].width = 14  # agent
        ws_c.column_dimensions["D"].width = 10  # role
        ws_c.column_dimensions["E"].width = 18  # content_type
        ws_c.column_dimensions["F"].width = 80  # content
        ws_c.column_dimensions["G"].width = 10  # meta_answer
        ws_c.column_dimensions["H"].width = 16  # meta_needs_revision
        ws_c.column_dimensions["I"].width = 30  # meta_issue
        ws_c.column_dimensions["J"].width = 30  # meta_action
        ws_c.column_dimensions["K"].width = 60  # meta_reasoning

        # ── wrap text برای ستون content ─────────────────
        from openpyxl.styles import Alignment
        for row_cells in ws_c.iter_rows(min_row=2):
            for cell in row_cells:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

        print(f"  ✅ sheet {sheet_name:15} → {len(df_conv)} ردیف")

print(f"\n🎉 Excel ذخیره شد: {EXCEL_PATH}")
print(f"   تعداد sheets: 2 (Summary + Accuracy) + {len(all_conv_rows)} (مکالمات)")

💾 در حال ذخیره Excel: F:\Thesis\project\3-Multi-Agent-System\Test_402\MA\eval_results\eval_402_20260627_203148.xlsx
  ✅ sheet Q01_conv        → 35 ردیف
  ✅ sheet Q02_conv        → 35 ردیف
  ✅ sheet Q03_conv        → 35 ردیف
  ✅ sheet Q04_conv        → 35 ردیف
  ✅ sheet Q05_conv        → 35 ردیف
  ✅ sheet Q06_conv        → 35 ردیف
  ✅ sheet Q07_conv        → 35 ردیف
  ✅ sheet Q08_conv        → 35 ردیف
  ✅ sheet Q09_conv        → 35 ردیف
  ✅ sheet Q10_conv        → 35 ردیف

🎉 Excel ذخیره شد: F:\Thesis\project\3-Multi-Agent-System\Test_402\MA\eval_results\eval_402_20260627_203148.xlsx
   تعداد sheets: 2 (Summary + Accuracy) + 10 (مکالمات)
